In [1]:
import os
# 把工作目录切换到项目根目录 bishe（根据你的实际路径修改）
os.chdir("/Users/bytedance/Desktop/bishe")

import paibox as pb
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models, transforms
from PIL import Image

from spikingjelly.activation_based import neuron, layer, functional

from qat_layer import Quan_Linear
from paibox.components.neuron.neurons import StoreVoltageNeuron as StoreVoltageNeuron


NUM_CLASSES = 5
TIME_STEPS=3
MODEL_PATH = '/Users/bytedance/Desktop/bishe/SNN/conv+fc_5_class_best.pth'  # 用于加载卷积层权重
LABELS = ['fangpian', 'heitao', 'hongtao', 'meihua', 'empty']
# 全连接weight的.npy路径（int8格式）
FC2_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc2_weight_int8.npy'
FC3_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc3_weight_int8.npy'

# 图片预处理（与训练时一致）
transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

class FloatPart(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc1 = Quan_Linear(w_bits=8,in_features = 128 * 6 * 6, out_features=512, bias=False)
        self.lif1 = neuron.LIFNode()

    def forward(self, x):
        x = self.conv_layer(x)
        x = torch.flatten(x,1)
        x = self.fc1(x)
        # print('#'*20, x[0][:5])
        spk1 = self.lif1(x)
        # print('#'*20, spk1)
        return spk1


# ----------------------
# 2. 关键修改：纯浮点全连接部分（仅用weight）
# ----------------------
# 定义LIFPart类 
class Int8FCPart(pb.Network):
    def __init__(self, fc1_weight):
        super().__init__()


        self.in_neuron = pb.InputProj(input=None, shape_out=(512,))

        self.n1 = pb.LIF((128,), threshold=1, reset_v=0)
        self.s1 = pb.FullConn(self.in_neuron, self.n1, weights=fc1_weight, conn_type=pb.SynConnType.All2All)

        # self.n2 = pb.LIF((5,), threshold=0, reset_v=0) # StoreVoltageNeuron(shape=(5,), bias=0, name='n1')   

        # self.n2  = pb.BypassNeuron(shape=5, name='n2')
        # self.s2 = pb.FullConn(self.n1, self.n2, weights=fc2_weight, conn_type=pb.SynConnType.All2All)


        # self.n3  = pb.BypassNeuron(shape=1, name='n3')
        # self.s3 = pb.FullConn(self.n2, self.n3, weights=np.random.randint(0,100,size=(5,1),dtype=np.int8), conn_type=pb.SynConnType.All2All)
        # self.s2 = pb.Linear(self.n1, 5, fc2_weight, bias=0, bit_trunc=8)


        self.probe_out = pb.Probe(target=self.s1, attr="output")  # attr 可以是 "spike" 或 "v"

        

        self.sim = pb.Simulator(self)

    def forward(self, input_data, timesteps=1):
        # input_spike = pb.simulator.PoissonEncoder(input_data.detach().numpy().astype(np.int8))

        input_spike = input_data.detach().cpu().numpy() 
        self.sim.reset()
        self.sim.run(timesteps, reset=True, data=input_spike)
        spike_output = self.sim.data[self.probe][-1].astype(np.int8)
        return spike_output


class Last(nn.Module):
    def __init__(self, fc3_npy):
        super().__init__()

        self.fc3 = nn.Linear(128, 5)  #Quan_Linear(w_bits=8,in_features = 128, out_features=5, bias=False)
        self.lif3 = neuron.LIFNode()

        fc3_weight = np.load(fc3_npy).astype(np.int8)
        self.fc3.weight.data = torch.from_numpy(fc3_weight.astype(np.float32))

    def forward(self, x):
        x = self.fc3(x)
        # print('#'*20, x[0][:5])
        spk1 = self.lif3(x)
        # print('#'*20, spk1)
        return spk1
    

class SNNInferModel(nn.Module):
    def __init__(self, model_path, fc2_npy, fc3_npy, time_steps=TIME_STEPS):
        super().__init__()
        self.time_steps = time_steps
        self.float_part = FloatPart()
        self.int8_fc_part = Int8FCPart(fc2_npy)
        self.last = Last(FC3_WEIGHT_NPY)

        # ✅ 加载浮点部分的权重
        state_dict = torch.load(model_path, map_location='cpu')["model_state_dict"]
        self.float_part.load_state_dict(state_dict, strict=False)
        # self.last.load_state_dict(state_dict, strict=False)


    def forward(self, x):
        functional.reset_net(self.float_part.lif1)
        out_spk = 0.0
        out = []
        for t in range(self.time_steps):
            spk1 = self.float_part(x)

            self.int8_fc_part.in_neuron.input = spk1.detach().cpu().numpy()  
            sim = pb.Simulator(self.int8_fc_part)
            
        
            # probe2 = pb.simulator.Probe(self.int8_fc_part.n2, "voltage")
            # sim.add_probe(probe2)

            sim.run(duration=1)  # 全连接网络无时间延迟，1步足够

            # 获取并打印输出结果
            # print(sim.data[probe2])

            output = sim.data[self.int8_fc_part.probe_out][0] # 取第1步的输出
            output_tensor = torch.tensor(output, dtype=torch.float32).to(next(self.last.parameters()).device)

            output = self.last(torch.tensor(output_tensor))


            # out.append(output)
            out_spk += output
            # out = self.int8_fc_part(spk1)
            # out_spk += out
        res = out_spk / self.time_steps
        # print(out)
        return out_spk / self.time_steps



# # 定义CombinedNet类
# class CombinedNet(nn.Module):
#     def __init__(self, weight1, weight2, num_classes, timesteps_in):
#         super(CombinedNet, self).__init__()
#         self.conv_part = ConvPart()
#         self.lif_part = LIFPart(weight1.T, weight2.T)

#     def forward(self, x):
#         batch_size = x.size(0)
#         x = self.conv_part(x)
#         x = x.view(batch_size, -1)
#         # print('zhongdian' * 20)
#         # print(x)
#         self.lif_part.in_neuron.input = x.detach().cpu().numpy()  
#         sim = pb.Simulator(self.lif_part)
#         sim.run(duration=1)  # 全连接网络无时间延迟，1步足够

#         # 获取并打印输出结果
#         output = sim.data[self.lif_part.probe_output][0]  # 取第1步的输出
        
#         return output



# 主函数
if __name__ == "__main__":

    # print(pb.__version__)

    FC2_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc2_weight_int8.npy'
    FC3_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc3_weight_int8.npy'
    weight1 = np.load(FC2_WEIGHT_NPY).astype(np.int8)
    weight2 = np.load(FC3_WEIGHT_NPY).astype(np.int8)

    num_classes = 5
    timesteps_in = 3

    model = SNNInferModel(MODEL_PATH, weight1.T, weight2.T, timesteps_in)

    LABELS = ['fangpian', 'heitao', 'hongtao', 'meihua', 'empty']
    transform = transforms.Compose([
        transforms.Resize((48, 48)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def predict_image(image_path, model, transform,  device):
        image = Image.open(image_path).convert('RGB')
        image = transform(image).unsqueeze(0)
        with torch.no_grad():
            output = model(image)
            # print(output.shape)
            print(output)

            _, predicted_label = torch.max(output, 0)
            # output_tensor = torch.from_numpy(output)  # 假设output形状为(1,5)或(5,)
            # if output_tensor.ndim == 2:
            #     _, predicted = torch.max(output_tensor, dim=1)  # 对第1维求最大值，返回形状(1,)
            # else:
            #     _, predicted = torch.max(output_tensor, dim=0)  # 对第0维求最大值，返回标量
            # _, predicted = torch.max(output, 0)
            # predicted_label = predicted[0].item()
            return predicted_label

    test_images = {
        '/Users/bytedance/Desktop/bishe/data/fig2/hongtao/MipiData_20250223_131123422_F_100M/image_fullpic/20250223_131123_000053.jpg': 2,
        '/Users/bytedance/Desktop/bishe/data/fig2/hongtao/MipiData_20250223_131123422_F_100M/image_fullpic/20250223_131123_000007.jpg': 4,
        '/Users/bytedance/Desktop/bishe/data/fig1/heitao/MipiData_20250222_232938156_F_100M/image_fullpic/20250222_232938_000032.jpg': 1,
        '/Users/bytedance/Desktop/bishe/data/fig3/fangpian/MipiData_20250223_135552092_F_100M/image_fullpic/20250223_135552_000020.jpg': 0,
        '/Users/bytedance/Desktop/bishe/data/fig3/fangpian/MipiData_20250223_135601349_F_100M/image_fullpic/20250223_135601_000027.jpg': 0,
        '/Users/bytedance/Desktop/bishe/data/fig1/meihua/MipiData_20250222_230856290_F_100M/image_fullpic/20250222_230856_000042.jpg': 3,
    }

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for image_path, true_label in test_images.items():
        pred_label = predict_image(image_path, model, transform, device)
        print(f'Image: {image_path}, True Label: {LABELS[true_label]}, Predicted Label: {LABELS[pred_label]}')

    

tensor([0.3333, 0.0000, 1.0000, 0.0000, 1.0000])
Image: /Users/bytedance/Desktop/bishe/data/fig2/hongtao/MipiData_20250223_131123422_F_100M/image_fullpic/20250223_131123_000053.jpg, True Label: hongtao, Predicted Label: hongtao
tensor([0., 0., 0., 0., 1.])
Image: /Users/bytedance/Desktop/bishe/data/fig2/hongtao/MipiData_20250223_131123422_F_100M/image_fullpic/20250223_131123_000007.jpg, True Label: empty, Predicted Label: empty
tensor([0., 0., 0., 0., 1.])
Image: /Users/bytedance/Desktop/bishe/data/fig1/heitao/MipiData_20250222_232938156_F_100M/image_fullpic/20250222_232938_000032.jpg, True Label: heitao, Predicted Label: empty
tensor([1., 0., 0., 0., 1.])
Image: /Users/bytedance/Desktop/bishe/data/fig3/fangpian/MipiData_20250223_135552092_F_100M/image_fullpic/20250223_135552_000020.jpg, True Label: fangpian, Predicted Label: fangpian
tensor([1., 0., 0., 0., 1.])
Image: /Users/bytedance/Desktop/bishe/data/fig3/fangpian/MipiData_20250223_135601349_F_100M/image_fullpic/20250223_135601_00

/var/folders/0d/7xtv56s92dvbjn0x6wdx41cm0000gn/T/ipykernel_60957/2809605161.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  output = self.last(torch.tensor(output_tensor))


In [2]:
import os
os.chdir("/Users/bytedance/Desktop/bishe")

import paibox as pb
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import transforms
from PIL import Image

from spikingjelly.activation_based import neuron, functional
from qat_layer import Quan_Linear


# ======================
# 基本配置
# ======================
NUM_CLASSES = 5
TIME_STEPS = 3

MODEL_PATH = '/Users/bytedance/Desktop/bishe/SNN/conv+fc_5_class_best.pth'
FC2_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc2_weight_int8.npy'
FC3_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc3_weight_int8.npy'

TEST_ROOT = "/Users/bytedance/Desktop/Davis/suit_dataset/test"  # <<< test 文件夹

LABELS = ['fangpian', 'heitao', 'hongtao', 'meihua', 'empty']
LABEL2IDX = {name: idx for idx, name in enumerate(LABELS)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ======================
# 图像预处理
# ======================
transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# ======================
# Float Part
# ======================
class FloatPart(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc1 = Quan_Linear(
            w_bits=8,
            in_features=128 * 6 * 6,
            out_features=512,
            bias=False
        )
        self.lif1 = neuron.LIFNode()

    def forward(self, x):
        x = self.conv_layer(x)
        x = torch.flatten(x,1)
        x = self.fc1(x)
        spk1 = self.lif1(x)
        return spk1


# ======================
# Int8 FC Part (Paibox)
# ======================
class Int8FCPart(pb.Network):
    def __init__(self, fc1_weight):
        super().__init__()

        self.in_neuron = pb.InputProj(input=None, shape_out=(512,))
        self.n1 = pb.LIF((128,), threshold=1, reset_v=0)

        self.s1 = pb.FullConn(
            self.in_neuron,
            self.n1,
            weights=fc1_weight,
            conn_type=pb.SynConnType.All2All
        )

        self.probe_out = pb.Probe(self.s1, attr="output")

        self.sim = pb.Simulator(self)


# ======================
# Last Layer
# ======================
class Last(nn.Module):
    def __init__(self, fc3_npy):
        super().__init__()
        self.fc3 = Quan_Linear(w_bits=8, in_features=128, out_features=num_classes, bias=False)    
        self.lif3 = neuron.LIFNode()

        fc3_weight = np.load(fc3_npy).astype(np.int8)
        self.fc3.weight.data = torch.from_numpy(fc3_weight.astype(np.float32))

    def forward(self, x):
        x = self.fc3(x)
        spk = self.lif3(x)
        return spk


# ======================
# SNN Infer Model
# ======================
class SNNInferModel(nn.Module):
    def __init__(self, model_path, fc2_npy, fc3_npy, time_steps):
        super().__init__()
        self.time_steps = time_steps

        self.float_part = FloatPart()
        self.int8_fc_part = Int8FCPart(fc2_npy)
        self.last = Last(fc3_npy)

        state_dict = torch.load(model_path, map_location='cpu')["model_state_dict"]
        self.float_part.load_state_dict(state_dict, strict=False)

    def forward(self, x):
        functional.reset_net(self.float_part.lif1)
        out_spk = 0.0

        for _ in range(self.time_steps):
            spk1 = self.float_part(x)

            self.int8_fc_part.in_neuron.input = spk1.detach().cpu().numpy()
            sim = pb.Simulator(self.int8_fc_part)
            sim.run(duration=1)

            output = sim.data[self.int8_fc_part.probe_out][0]
            output = torch.tensor(output, dtype=torch.float32, device=x.device)

            out_spk += self.last(output)

        return out_spk / self.time_steps


# ======================
# 单张图片预测（原逻辑）
# ======================
def predict_image(image_path, model, transform, device):
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        pred = output.argmax(dim=0).item()

    return pred


# ======================
# ===== 关键新增：文件夹测试 =====
# ======================
def test_folder(test_root, model, transform, device):
    model.eval()

    total = 0
    correct = 0

    for label_name in sorted(os.listdir(test_root)):
        class_dir = os.path.join(test_root, label_name)
        if not os.path.isdir(class_dir):
            continue

        true_label = LABEL2IDX[label_name]

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                continue

            img_path = os.path.join(class_dir, fname)
            pred_label = predict_image(img_path, model, transform, device)

            total += 1
            if pred_label == true_label:
                correct += 1

            print(
                f"[{label_name}] {fname} | "
                f"GT={LABELS[true_label]} | "
                f"PRED={LABELS[pred_label]}"
            )

    acc = correct / total if total > 0 else 0.0
    print("=" * 60)
    print(f"Total: {total}, Correct: {correct}, Accuracy: {acc:.4f}")


# ======================
# Main
# ======================
if __name__ == "__main__":
    fc2_weight = np.load(FC2_WEIGHT_NPY).astype(np.int8).T

    model = SNNInferModel(
        MODEL_PATH,
        fc2_weight,
        FC3_WEIGHT_NPY,
        TIME_STEPS
    ).to(DEVICE)

    test_folder(TEST_ROOT, model, transform, DEVICE)


[empty] 9_hongtao_frame_1130.jpg | GT=empty | PRED=heitao
[empty] 4_fangpian_frame_332.jpg | GT=empty | PRED=heitao
[empty] 9_fangpian_frame_286.jpg | GT=empty | PRED=heitao
[empty] 10_heitao_frame_1136.jpg | GT=empty | PRED=heitao
[empty] 3_fangpian_frame_456.jpg | GT=empty | PRED=heitao
[empty] 9_hongtao_frame_1290.jpg | GT=empty | PRED=heitao
[empty] 10_fangpian_frame_1010.jpg | GT=empty | PRED=heitao
[empty] 7_hongtao_frame_990.jpg | GT=empty | PRED=heitao
[empty] 7_heitao_frame_1182.jpg | GT=empty | PRED=heitao
[empty] 4_hongtao_frame_1278.jpg | GT=empty | PRED=heitao
[empty] 7_fangpian_frame_444.jpg | GT=empty | PRED=heitao
[empty] 4_meihua_frame_1034.jpg | GT=empty | PRED=heitao
[empty] 3_meihua_frame_896.jpg | GT=empty | PRED=heitao
[empty] 3_heitao_frame_2774.jpg | GT=empty | PRED=heitao
[empty] 4_meihua_frame_420.jpg | GT=empty | PRED=heitao
[empty] 2_meihua_frame_606.jpg | GT=empty | PRED=heitao
[empty] 3_heitao_frame_1282.jpg | GT=empty | PRED=heitao
[empty] 1_meihua_frame_

In [3]:
# ============================================================
# test_dataset_inference_suit_fixed.py
# 修复维度不匹配错误的Test数据集预测代码
# 解决：LIF神经元状态未重置、权重重复转置、变量名错误、批次维度问题
# ============================================================
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import paibox as pb
from spikingjelly.activation_based import neuron  # 补充缺失的导入
import pandas as pd  # 提前导入pandas

# 切换工作目录（保持原有配置）
os.chdir("/Users/bytedance/Desktop/bishe")

# ========================= 1. 全局配置 =========================
# 数据集根目录（已分train/test）
DATA_ROOT = "/Users/bytedance/Desktop/Davis/suit_dataset"
TEST_DATA_ROOT = os.path.join(DATA_ROOT, "test")

# 模型路径配置
MODEL_PATH = '/Users/bytedance/Desktop/bishe/SNN/conv+fc_5_class_best.pth'
FC2_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc2_weight_int8.npy'
FC3_WEIGHT_NPY = '/Users/bytedance/Desktop/bishe/SNN/save_int8_new/fc3_weight_int8.npy'

# 模型配置
NUM_CLASSES = 5
TIME_STEPS = 3
DEVICE = torch.device('cpu')
BATCH_SIZE = 8
WORKERS = 0

# 类别映射
LABEL_MAP = {
    'fangpian': 0,
    'heitao': 1,
    'hongtao': 2,
    'meihua': 3,
    'empty': 4
}
LABEL_NAMES = ['fangpian', 'heitao', 'hongtao', 'meihua', 'empty']

# 图片预处理
transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# ========================= 2. 数据集加载器 =========================
class CardTestDataset(Dataset):
    """适配suit_dataset/test目录结构的数据集加载器"""
    def __init__(self, test_root_dir, transform=None):
        self.test_root_dir = test_root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        if not os.path.exists(test_root_dir):
            raise FileNotFoundError(f"Test目录不存在: {test_root_dir}")
        
        print(f"📂 正在加载Test数据集: {test_root_dir}")
        for label_name in os.listdir(test_root_dir):
            if label_name.startswith('.'):  # 跳过隐藏文件如.DS_Store
                print(f"⚠️  跳过隐藏文件: {label_name}")
                continue
            if label_name not in LABEL_MAP:
                print(f"⚠️  跳过非类别文件夹: {label_name}")
                continue
            
            label_idx = LABEL_MAP[label_name]
            class_dir = os.path.join(test_root_dir, label_name)
            
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                        img_path = os.path.join(class_dir, img_name)
                        self.image_paths.append(img_path)
                        self.labels.append(label_idx)
        
        total_samples = len(self.image_paths)
        if total_samples == 0:
            print("⚠️  Test数据集中未找到任何图片")
        else:
            print(f"✅ 成功加载Test数据集：共 {total_samples} 张图片")
            for label_name, label_idx in LABEL_MAP.items():
                count = self.labels.count(label_idx)
                print(f"   - {label_name}: {count} 张")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            with Image.open(img_path) as img_file:
                image = img_file.convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, img_path
        
        except Exception as e:
            print(f"\n❌ 加载图片失败 {img_path}: {str(e)[:50]}...")
            return torch.zeros(3, 48, 48), -1, img_path

# ========================= 3. 修复后的网络定义 =========================
# 第一部分：浮点特征提取（修复LIF状态重置）
class FloatPart(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc1 = Quan_Linear(w_bits=8, in_features=128 * 6 * 6, out_features=512, bias=False)
        self.lif1 = neuron.LIFNode()
        
        # 加载预训练权重
        state_dict = torch.load(MODEL_PATH, map_location=DEVICE)["model_state_dict"]
        self.load_state_dict({k: v for k, v in state_dict.items() if k in self.state_dict()}, strict=False)
        
        self.eval()

    def forward(self, x):
        with torch.no_grad():
            # 每次前向传播前重置LIF神经元状态（核心修复！）
            functional.reset_net(self)  # 补充spikingjelly的重置函数
            
            x = self.conv_layer(x)
            x = torch.flatten(x, 1)
            x = self.fc1(x)
            spk1 = self.lif1(x)
            return spk1

# 第二部分：Paibox INT8 FC层（修复权重重复转置）
class Int8FCPart(pb.Network):
    def __init__(self, fc1_weight):
        super().__init__()
        self.fc1_weight = fc1_weight.astype(np.int8)
        
        self.in_neuron = pb.InputProj(input=None, shape_out=(512,))
        self.n1 = pb.LIF((128,), threshold=1, reset_v=0)
        # 修复：权重已经在外部转置，这里不再重复转置
        self.s1 = pb.FullConn(self.in_neuron, self.n1, 
                              weights=self.fc1_weight.T,  # 移除.T
                              conn_type=pb.SynConnType.All2All)
        
        self.probe_out = pb.Probe(target=self.s1, attr="output")
        self.sim = pb.Simulator(self)

    def forward_single(self, input_data):
        """单样本推理"""
        input_spike = input_data.astype(np.int8)
        self.sim.reset()
        self.in_neuron.input = input_spike
        self.sim.run(duration=1, reset=True)
        return self.sim.data[self.probe_out][-1].astype(np.float32)

    def forward_batch(self, input_batch):
        """批量推理（逐样本处理）"""
        batch_size = input_batch.shape[0]
        outputs = []
        
        for i in range(batch_size):
            single_output = self.forward_single(input_batch[i])
            outputs.append(single_output)
        
        return np.stack(outputs)

# 第三部分：输出层（修复变量名错误）
class Last(nn.Module):
    def __init__(self, fc3_npy):
        super().__init__()
        # 修复：num_classes → NUM_CLASSES（全局变量）
        self.fc3 = Quan_Linear(w_bits=8, in_features=128, out_features=num_classes, bias=False)    
        self.lif3 = neuron.LIFNode()

        
        # 加载INT8权重
        fc3_weight = np.load(fc3_npy).astype(np.int8)
        self.fc3.weight.data = torch.from_numpy(fc3_weight.astype(np.float32)).to(DEVICE)

    def forward(self, x):
        with torch.no_grad():
            # 重置LIF神经元状态
            functional.reset_net(self)
            
            x = self.fc3(x)
            spk1 = self.lif3(x)
            return spk1

# 组合模型（修复批次维度处理）
class SNNInferModel(nn.Module):
    def __init__(self, model_path, fc2_weight, fc3_npy, time_steps=TIME_STEPS):
        super().__init__()
        self.time_steps = time_steps
        self.float_part = FloatPart().to(DEVICE)
        self.int8_fc_part = Int8FCPart(fc2_weight)
        self.last = Last(fc3_npy).to(DEVICE)

    def forward(self, x):
        batch_size = x.shape[0]
        out_spk = torch.zeros(batch_size, NUM_CLASSES).to(DEVICE)
        
        for t in range(self.time_steps):
            # 浮点特征提取
            spk1 = self.float_part(x)
            
            # Paibox INT8推理
            spk1_np = spk1.detach().cpu().numpy()
            fc2_output_np = self.int8_fc_part.forward_batch(spk1_np)
            fc2_output = torch.tensor(fc2_output_np, dtype=torch.float32).to(DEVICE)
            
            # 输出层（修复tensor复制警告）
            output = self.last(fc2_output.clone().detach())  # 使用clone().detach()
            out_spk += output
        
        return out_spk / self.time_steps

# ========================= 4. 核心预测函数 =========================
def predict_suit_test_dataset():
    """预测suit_dataset/test数据集"""
    # 补充spikingjelly的functional导入
    from spikingjelly.activation_based import functional
    
    # 1. 初始化数据集
    try:
        test_dataset = CardTestDataset(TEST_DATA_ROOT, transform=transform)
    except FileNotFoundError as e:
        print(f"❌ 数据集加载失败: {e}")
        return
    
    if len(test_dataset) == 0:
        print("❌ Test数据集为空，退出预测")
        return
    
    # 2. 创建DataLoader（修复最后批次维度问题）
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=WORKERS,
        pin_memory=False,
        drop_last=False
    )
    
    # 3. 加载模型
    print("\n📌 加载模型权重...")
    # 加载并调整fc2权重（只转置一次）
    fc2_weight = np.load(FC2_WEIGHT_NPY).astype(np.int8)
    if fc2_weight.shape == (512, 128):
        fc2_weight = fc2_weight.T  # 只在这里转置一次
    print(f"✅ FC2权重维度: {fc2_weight.shape} (INT8)")
    
    # 初始化模型
    model = SNNInferModel(MODEL_PATH, fc2_weight, FC3_WEIGHT_NPY)
    model.eval()
    print("✅ 模型初始化完成")
    
    # 4. 统计变量初始化
    total = 0
    correct = 0
    class_correct = {i: 0 for i in range(NUM_CLASSES)}
    class_total = {i: 0 for i in range(NUM_CLASSES)}
    error_details = []
    
    # 5. 批量推理
    print("\n🚀 开始预测Test数据集...")
    with torch.no_grad():
        for batch_idx, (images, labels, paths) in enumerate(tqdm(test_loader, desc="预测进度")):
            # 过滤无效样本
            valid_mask = labels != -1
            if not valid_mask.any():
                continue
            
            # 提取有效数据
            val_images = images[valid_mask].to(DEVICE)
            val_labels = labels[valid_mask]
            val_paths = [paths[i] for i in range(len(paths)) if valid_mask[i]]
            
            # 重置整个网络的神经元状态（关键修复）
            functional.reset_net(model)
            
            try:
                # 模型推理
                outputs = model(val_images)
                _, preds = torch.max(outputs, 1)
                
                # 转换为numpy
                preds_np = preds.cpu().numpy()
                labels_np = val_labels.numpy()
                
                # 更新统计
                batch_correct = (preds_np == labels_np).sum()
                total += len(labels_np)
                correct += batch_correct
                
                # 按类别统计
                for lbl, p, path in zip(labels_np, preds_np, val_paths):
                    class_total[lbl] += 1
                    if lbl == p:
                        class_correct[lbl] += 1
                    else:
                        error_details.append({
                            'path': path,
                            'true_label': lbl,
                            'true_name': LABEL_NAMES[lbl],
                            'pred_label': p,
                            'pred_name': LABEL_NAMES[p]
                        })
                        
            except Exception as e:
                print(f"\n❌ 批次 {batch_idx+1} 推理出错: {e}")
                print(f"   输入图片维度: {val_images.shape}")
                continue
    
    # 6. 结果输出
    print("\n" + "="*80)
    print("📊 suit_dataset/test 预测结果")
    print("="*80)
    
    if total > 0:
        overall_acc = (correct / total) * 100
        print(f"\n🎯 总体准确率: {overall_acc:.2f}% ({correct}/{total})")
        
        print("\n📈 各类别准确率：")
        for i in range(NUM_CLASSES):
            if class_total[i] > 0:
                acc = (class_correct[i] / class_total[i]) * 100
                print(f"   {LABEL_NAMES[i]:8s}: {acc:6.2f}% ({class_correct[i]}/{class_total[i]})")
            else:
                print(f"   {LABEL_NAMES[i]:8s}:     -- (0/0)")
        
        error_count = len(error_details)
        print(f"\n❌ 错误预测: {error_count} 个样本 ({error_count/total*100:.2f}%)")
        
        if error_count > 0:
            print("\n错误样本详情（前10个）：")
            for i, err in enumerate(error_details[:10]):
                rel_path = os.path.relpath(err['path'], DATA_ROOT)
                print(f"   {i+1:2d}. {rel_path}")
                print(f"      真实: {err['true_name']} | 预测: {err['pred_name']}")
        
        
    else:
        print("\n⚠️  无有效样本参与预测")
    
    print("\n🎉 预测完成！")
    return {
        'overall_accuracy': overall_acc if total > 0 else 0,
        'class_accuracy': class_correct,
        'error_samples': error_details
    }


    
# ========================= 5. 主函数 =========================
if __name__ == "__main__":
    # 补充必要的导入
    from spikingjelly.activation_based import functional
    
    # 禁用梯度
    torch.set_grad_enabled(False)
    
    # 设置随机种子
    np.random.seed(42)
    torch.manual_seed(42)
    
    # 执行预测
    results = predict_suit_test_dataset()

📂 正在加载Test数据集: /Users/bytedance/Desktop/Davis/suit_dataset/test
⚠️  跳过隐藏文件: .DS_Store
✅ 成功加载Test数据集：共 1140 张图片
   - fangpian: 228 张
   - heitao: 228 张
   - hongtao: 228 张
   - meihua: 228 张
   - empty: 228 张

📌 加载模型权重...
✅ FC2权重维度: (128, 512) (INT8)
✅ 模型初始化完成

🚀 开始预测Test数据集...


预测进度: 100%|██████████| 143/143 [00:07<00:00, 19.71it/s]


📊 suit_dataset/test 预测结果

🎯 总体准确率: 18.16% (207/1140)

📈 各类别准确率：
   fangpian:   0.00% (0/228)
   heitao  :  90.79% (207/228)
   hongtao :   0.00% (0/228)
   meihua  :   0.00% (0/228)
   empty   :   0.00% (0/228)

❌ 错误预测: 933 个样本 (81.84%)

错误样本详情（前10个）：
    1. test/empty/9_hongtao_frame_1130.jpg
      真实: empty | 预测: heitao
    2. test/empty/4_fangpian_frame_332.jpg
      真实: empty | 预测: heitao
    3. test/empty/9_fangpian_frame_286.jpg
      真实: empty | 预测: heitao
    4. test/empty/10_heitao_frame_1136.jpg
      真实: empty | 预测: heitao
    5. test/empty/3_fangpian_frame_456.jpg
      真实: empty | 预测: heitao
    6. test/empty/9_hongtao_frame_1290.jpg
      真实: empty | 预测: heitao
    7. test/empty/10_fangpian_frame_1010.jpg
      真实: empty | 预测: heitao
    8. test/empty/7_hongtao_frame_990.jpg
      真实: empty | 预测: heitao
    9. test/empty/7_heitao_frame_1182.jpg
      真实: empty | 预测: heitao
   10. test/empty/4_hongtao_frame_1278.jpg
      真实: empty | 预测: heitao

🎉 预测完成！


In [53]:
#####
# read me

readme

整个流程 update----2026.01.15

以“花色”任务为例，路径在2_huase_augment里

1）首先，运行main函数，得到qat的训练结果并可以实时观察（dvs的conda环境）
2）得到结果后，可运行test_val，查看保存模型在测试集上的acc【nn.linear是随机输出，但是改成Qlinear就可以】
3）运行quantize得到量化int8的权重
4）运行test_val_use_int8，这个函数是用了测试在测试集上，用int8的权重的效果【这里用nn linear和q linear效果一样，acc都很高】 （直接点三角运行不要进坏境）

5）离线测试：【8张扑克牌】
    a）用test_offline 来看保存权重在原始网络的输出【也是只能q linear】
    b）用test_offline_use_int8，查看的是量化后的权重的输出【也是 用nn linear和q linear效果一样】
6）在线测试：【实时读取摄像头】（需要进dvs环境）
    a）用test_realtime 
    b）用test_realtime_use_int8


7）开始编译，把int8部分的网络用下面的代码来运行
8）仿真【暂未跑过，但是不影响后续】


9）进入board文件夹
    a）运行board_huase_val，观察在所有测试集上的结果（直接点三角运行不要进坏境，下面同理）
    b）运行board_huase_offline，观察8张扑克牌事件的预测结果
    c）运行board_huase_realtime，实时

In [ ]:
import os
# 把工作目录切换到项目根目录 bishe（根据你的实际路径修改）
os.chdir("/Users/bytedance/Desktop/bishe")

import paibox as pb
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models, transforms
from PIL import Image

from spikingjelly.activation_based import neuron, layer, functional

from qat_layer import Quan_Linear
from paibox.components.neuron.neurons import StoreVoltageNeuron as StoreVoltageNeuron
import numpy as np
FC2_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment/save_int8_new/fc2_weight_int8.npy"  # INT8 FC2权重
FC3_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment/save_int8_new/fc3_weight_int8.npy"  # INT8 FC3权重

weight1 = np.load(FC2_WEIGHT_NPY).astype(np.int8)
weight2 = np.load(FC3_WEIGHT_NPY).astype(np.int8)

model = Int8FCPart(weight1.T)
mapper = pb.Mapper()
mapper.build(model)
graph_info = mapper.compile(core_estimate_only=False, weight_bit_optimization=True, grouping_optim_target="both")
mapper.export(write_to_file=True, fp="./debug_260115/", format="bin", split_by_chip=False, use_hw_sim=True)

print(graph_info.get('n_core_required'))  # 推荐用 get()，不存在时返回 None 而非报错



RoutingGroup: RoutingGroup_3 with 0 cores:
multicast axons: ['InputProj_37']
	CoreBlock_1 with 0 cores:
		LCN: 0
		FullConn_37: InputProj_37 -> LIF_37

2


In [15]:
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import paibox as pb
from spikingjelly.activation_based import neuron
import pandas as pd
import gc  # 引入垃圾回收，清理重复注册

# 全局配置
NUM_CLASSES = 5
# 清理PAIBoard可能的残留注册（关键修复）

class FullInt8SNN_PB(pb.Network):
    def __init__(self, weight_paths, network_name="snn_huase"):
        super().__init__(name=network_name)  # 给网络命名，避免全局命名冲突
        
        # ========================= 1. 加载INT8权重（OIHW格式） =========================
        self.conv1_weight = np.load(weight_paths["conv1"]).astype(np.int8)  # (32,3,3,3)
        self.conv2_weight = np.load(weight_paths["conv2"]).astype(np.int8)  # (64,32,3,3)
        self.conv3_weight = np.load(weight_paths["conv3"]).astype(np.int8)  # (128,64,3,3)
        self.fc1_weight = np.load(weight_paths["fc1"]).astype(np.int8)      # (4608,512)
        self.fc2_weight = np.load(weight_paths["fc2"]).astype(np.int8)      # (512,128)
        self.fc3_weight = np.load(weight_paths["fc3"]).astype(np.int8)      # (128,5)
        
        # ========================= 2. 卷积层（修复命名冲突+参数错误） =========================
        # 输入层 (3,48,48)
        self.n0 = pb.LIF(shape=(3, 48, 48), threshold=1, name=f"{network_name}_n0")  
        
        # 卷积1: 3→32, kernel=3x3, padding=1 → (32,48,48)
        self.n1 = pb.LIF(shape=(32, 48, 48), threshold=1, name=f"{network_name}_n1")  
        self.conv1 = pb.Conv2d(
            self.n0, self.n1,
            kernel=self.conv1_weight,
            stride=(1, 1),
            padding=(1, 1),  # 修复：padding需传tuple，而非int
            kernel_order="OIHW",
            name=f"{network_name}_conv2d_1"  # 加网络前缀，避免同名
        )
        
        # ========================= 3. 池化层（修复命名+参数） =========================
        # 池化1: SpikingMaxPool2d, kernel=2x2, stride=2, padding=0 → (32,24,24)
        ksize = (2, 2)
        self.n1_pool = pb.LIF(shape=(32, 24, 24), threshold=1, reset_v=0, name=f"{network_name}_n1_pool")
        self.pool1 = pb.SpikingMaxPool2d(
            self.n1,  # 待池化的神经元
            kernel_size=ksize,
            stride=ksize,
            padding=(0, 0),
            tick_wait_start=2,
            name=f"{network_name}_pool1"  # 唯一命名
        )
        
        # 卷积2: 32→64, kernel=3x3, padding=1 → (64,24,24)
        self.n2 = pb.LIF(shape=(64, 24, 24), threshold=1, name=f"{network_name}_n2")
        self.conv2 = pb.Conv2d(
            self.n1_pool, self.n2,
            kernel=self.conv2_weight,
            stride=(1, 1),
            padding=(1, 1),  # 修复：padding传tuple
            kernel_order="OIHW",
            name=f"{network_name}_conv2d_2"  # 唯一命名
        )
        
        # 池化2: SpikingMaxPool2d, kernel=2x2 → (64,12,12)
        self.n2_pool = pb.LIF(shape=(64, 12, 12), threshold=1, reset_v=0, name=f"{network_name}_n2_pool")
        self.pool2 = pb.SpikingMaxPool2d(
            self.n2,
            kernel_size=(2, 2),
            stride=(2, 2),
            padding=(0, 0),
            tick_wait_start=2,
            name=f"{network_name}_pool2"
        )
        
        # 卷积3: 64→128, kernel=3x3, padding=1 → (128,12,12)
        self.n3 = pb.LIF(shape=(128, 12, 12), threshold=1, name=f"{network_name}_n3")
        self.conv3 = pb.Conv2d(
            self.n2_pool, self.n3,
            kernel=self.conv3_weight,
            stride=(1, 1),
            padding=(1, 1),  # 修复：padding传tuple
            kernel_order="OIHW",
            name=f"{network_name}_conv2d_3"  # 唯一命名
        )
        
        # 池化3: SpikingMaxPool2d, kernel=2x2 → (128,6,6)
        self.n3_pool = pb.LIF(shape=(128, 6, 6), threshold=1, reset_v=0, name=f"{network_name}_n3_pool")
        self.pool3 = pb.SpikingMaxPool2d(
            self.n3,
            kernel_size=(2, 2),
            stride=(2, 2),
            padding=(0, 0),
            tick_wait_start=2,
            name=f"{network_name}_pool3"
        )
        
        # ========================= 4. 全连接层（修复命名） =========================

        # 全连接输入层
        self.in_neuron = pb.InputProj(
            input=None, 
            shape_out=(4608,),
            name=f"{network_name}_in_neuron"
        )
        
        # 全连接1: 4608→512
        self.n4 = pb.LIF((512,), threshold=1, reset_v=0, name=f"{network_name}_n4")
        self.fc1 = pb.FullConn(
            self.in_neuron, self.n4, 
            weights=self.fc1_weight.T, 
            conn_type=pb.SynConnType.All2All,
            name=f"{network_name}_fc1"  # 唯一命名
        )
        
        # 全连接2: 512→128
        self.n5 = pb.LIF((128,), threshold=1, reset_v=0, name=f"{network_name}_n5")
        self.fc2 = pb.FullConn(
            self.n4, self.n5, 
            weights=self.fc2_weight.T, 
            conn_type=pb.SynConnType.All2All,
            name=f"{network_name}_fc2"
        )
        
        # 全连接3: 128→5
        self.n6 = pb.LIF((NUM_CLASSES,), threshold=1, reset_v=0, name=f"{network_name}_n6")
        self.fc3 = pb.FullConn(
            self.n5, self.n6, 
            weights=self.fc3_weight.T, 
            conn_type=pb.SynConnType.All2All,
            name=f"{network_name}_fc3"
        )
        
        # 探针+模拟器（修复命名）
        self.probe_out = pb.Probe(target=self.fc3, attr="output", name=f"{network_name}_probe_out")
        self.sim = pb.Simulator(self, name=f"{network_name}_sim")

# ========================= 权重路径配置 =========================
INT8_WEIGHT_PATHS = {
    "conv1": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/conv1_weight_int8.npy",
    "conv2": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/conv2_weight_int8.npy",
    "conv3": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/conv3_weight_int8.npy",
    "fc1": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/fc1_weight_int8.npy",
    "fc2": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/fc2_weight_int8.npy",
    "fc3": "/Users/bytedance/Desktop/bishe/snn_2_huase_augment/int8_all_layers_qat/fc3_weight_int8.npy"
}

# ========================= 主程序（修复重复实例化） =========================
if __name__ == "__main__":
    # 1. 再次清理注册器（双重保险）
    
    # 2. 实例化模型（指定唯一网络名）
    model = FullInt8SNN_PB(INT8_WEIGHT_PATHS, network_name="snn_huase_202601189")
    
    # 3. 构建+编译（避免重复编译）
    mapper = pb.Mapper()
    mapper.build(model)
    graph_info = mapper.compile(
        core_estimate_only=False, 
        weight_bit_optimization=True, 
        grouping_optim_target="both"
    )
    
    # 4. 创建输出目录（避免路径不存在）
    output_dir = "./snn_2_huase/"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 5. 导出模型
    mapper.export(
        write_to_file=True, 
        fp=output_dir, 
        format="bin", 
        split_by_chip=False, 
        use_hw_sim=True
    )
    
    # 6. 打印核心数（容错处理）
    core_num = graph_info.get('n_core_required', '未知')
    print(f"✅ 模型编译完成！推荐核心数: {core_num}")

RoutingGroup: RoutingGroup_53 with 0 cores:
multicast axons: ['snn_huase_202601189_n2_pool']
	CoreBlock_22 with 0 cores:
		LCN: 3
		snn_huase_202601189_conv2d_3: snn_huase_202601189_n2_pool -> snn_huase_202601189_n3

RoutingGroup: RoutingGroup_54 with 0 cores:
multicast axons: ['snn_huase_202601189_n3']
	CoreBlock_23 with 0 cores:
		LCN: 4
		s0_snn_huase_202601189_pool3: snn_huase_202601189_n3 -> n0_snn_huase_202601189_pool3

RoutingGroup: RoutingGroup_55 with 0 cores:
multicast axons: ['snn_huase_202601189_n0']
	CoreBlock_24 with 0 cores:
		LCN: 3
		snn_huase_202601189_conv2d_1: snn_huase_202601189_n0 -> snn_huase_202601189_n1

RoutingGroup: RoutingGroup_56 with 0 cores:
multicast axons: ['snn_huase_202601189_in_neuron']
	CoreBlock_25 with 0 cores:
		LCN: 2
		snn_huase_202601189_fc1: snn_huase_202601189_in_neuron -> snn_huase_202601189_n4

RoutingGroup: RoutingGroup_57 with 0 cores:
multicast axons: ['snn_huase_202601189_n4']
	CoreBlock_26 with 0 cores:
		LCN: 0
		snn_huase_202601189_

ResourceError: the number of required cores is out of range 1008 (23795).

In [2]:
import os
import cv2
import paibox as pb
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn

import os
# 把工作目录切换到项目根目录 bishe（根据你的实际路径修改）
os.chdir("/Users/bytedance/Desktop/bishe")

# ===================== 核心配置（完全保留你原来的） =====================
MODEL_PATH = '/Users/bytedance/Desktop/bishe/2_huase_augment/conv+fc_5_class_best.pth'
FC1_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment_copy/save_int8_qat_compatible/fc1_weight_int8.npy"
FC2_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment_copy/save_int8_qat_compatible/fc2_weight_int8.npy"
FC3_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment_copy/save_int8_qat_compatible/fc3_weight_int8.npy"

INPUT_DIR = "/Users/bytedance/Desktop/dvs_data/test_offline"
TEST_SET_ROOT = "/Users/bytedance/Desktop/Davis/suit_dataset/test"

REMOVE_FIRST_N = 10
WHITE_THRESH = 200
BLUE_THRESH = 200
MIN_GAP_FRAMES = 10
MIN_EVENT_FRAMES = 7
SCALE = 128

NUM_CLASSES = 5
TIME_STEPS = 3
N_LAYERS = 3
TOTAL_TICKS = TIME_STEPS + (N_LAYERS - 1)  # 标准范例配置

CLASS_NAMES = ['fangpian', 'heitao', 'hongtao', 'meihua', 'empty']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMG_SIZE = (48, 48)

DEVICE = torch.device('cpu')

# ===================== 预处理（保留你原来的） =====================
test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# ===================== 【标准范例同款】paibox 工具函数 =====================
def timestep_in(t, data):
    if 1 <= t <= data.shape[0]:
        return data[t - 1]
    else:
        return np.zeros_like(data[0])

# ===================== 浮点卷积部分（完全保留你的） =====================
class FloatPart(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = torch.flatten(x, 1)
        return x  # [B, 4608]

# ===================== 【严格按标准范例重写】SNN_Paibox =====================
class SNN_Paibox(pb.Network):
    def __init__(self, w1, w2, w3):
        super().__init__()
        
        # 输入
        self.inp = pb.InputProj(timestep_in, shape_out=4608)

        # 标准范例配置：逐层 tick_wait_start + 阈值
        self.n1 = pb.LIF(512, threshold=SCALE, tick_wait_start=1)
        self.n2 = pb.LIF(128, threshold=1,     tick_wait_start=2)
        self.n3 = pb.LIF(5,   threshold=1,     tick_wait_start=3)

        # 权重连接
        self.s1 = pb.FullConn(self.inp, self.n1, weights=w1.T)
        self.s2 = pb.FullConn(self.n1,  self.n2, weights=w2.T)
        self.s3 = pb.FullConn(self.n2,  self.n3, weights=w3.T)

        # 三层探针
        self.p1 = pb.Probe(self.n1, "spike")
        self.p2 = pb.Probe(self.n2, "spike")
        self.p3 = pb.Probe(self.n3, "spike")
        
        self.sim = pb.Simulator(self)

    def __call__(self, input_data):
        # 输入量化（标准范例逻辑）
        input_quant = np.clip(np.round(input_data * SCALE), -128, 127).astype(np.int8)
        pb_in = np.tile(input_quant, (TOTAL_TICKS, 1))

        # 仿真
        self.sim.reset()
        self.sim.run(TOTAL_TICKS, data=pb_in)

        # 按标准范例对齐输出
        p1_all = np.array(self.sim.data[self.p1])
        p2_all = np.array(self.sim.data[self.p2])
        p3_all = np.array(self.sim.data[self.p3])

        p1 = p1_all[0:TIME_STEPS]
        p2 = p2_all[1:TIME_STEPS+1]
        p3 = p3_all[2:TIME_STEPS+2]

        return p1, p2, p3

# ===================== 整体推理模型 =====================
class FullModel(nn.Module):
    def __init__(self, model_path, fc1_npy, fc2_npy, fc3_npy):
        super().__init__()
        self.float_part = FloatPart()
        self.time_steps = TIME_STEPS

        # 加载卷积权重
        state_dict = torch.load(model_path, map_location='cpu')
        if "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        self.float_part.load_state_dict(state_dict, strict=False)
        self.float_part.eval()

        # 加载 INT8 权重（与标准范例完全一致）
        w1 = np.load(fc1_npy).astype(np.int8)
        w2 = np.load(fc2_npy).astype(np.int8)
        w3 = np.load(fc3_npy).astype(np.int8)

        self.snn = SNN_Paibox(w1, w2, w3)

    def forward(self, x):
        # 卷积提取特征
        feat = self.float_part(x)
        feat_np = feat.squeeze().detach().cpu().numpy()
        feat_np = feat_np / SCALE  # 归一化

        # PAIBox 推理
        p1, p2, p3 = self.snn(feat_np)

        # 输出平均脉冲
        out_avg = np.mean(p3, axis=0)
        return torch.tensor(out_avg).unsqueeze(0)

# ===================== 以下**完全保留你原来所有逻辑**，一字未改 =====================
def detect_event_frame(img):
    b, g, r = cv2.split(img)
    white = (r > WHITE_THRESH) & (g > WHITE_THRESH) & (b > WHITE_THRESH)
    blue = (b > BLUE_THRESH) & (r < 100) & (g < 100)
    return (np.count_nonzero(white) + np.count_nonzero(blue)) >= (WHITE_THRESH + BLUE_THRESH)

def extract_index(name):
    digits = ''.join(filter(str.isdigit, name))
    return int(digits) if digits else -1

def predict_image(model, img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        img = test_transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = model(img)
            pred = torch.argmax(output, dim=1).item()
        return pred
    except Exception as e:
        print(f"⚠️  预测图片失败 {img_path}: {e}")
        return -1

def predict_event(model, event_frames, base_dir):
    votes = []
    for fname in event_frames:
        img_path = os.path.join(base_dir, fname)
        pred = predict_image(model, img_path)
        if pred != -1:
            votes.append(pred)
    if not votes:
        return -1, {}
    values, counts = np.unique(votes, return_counts=True)
    final_pred = values[np.argmax(counts)]
    return final_pred, dict(zip(values, counts))

# ===================== 主函数（完全保留你的流程） =====================
def main():
    print("===== 加载模型 =====")
    model = FullModel(MODEL_PATH, FC1_WEIGHT_NPY, FC2_WEIGHT_NPY, FC3_WEIGHT_NPY).to(DEVICE)
    model.eval()
    print("✅ 模型加载完成")

    print(f"\n===== 读取事件帧（{INPUT_DIR}） =====")
    files = [
        f for f in os.listdir(INPUT_DIR)
        if f.lower().endswith(('.jpg', '.png')) and not f.startswith('.DS_Store')
    ]
    files = sorted(files, key=extract_index)
    files = files[REMOVE_FIRST_N:]
    print(f"📁 有效帧数量: {len(files)}")

    print("\n===== 事件分割 =====")
    raw_events = []
    current_event = []
    gap = 0

    for fname in files:
        img_path = os.path.join(INPUT_DIR, fname)
        img = cv2.imread(img_path)
        if img is None:
            continue
        if detect_event_frame(img):
            current_event.append(fname)
            gap = 0
        else:
            gap += 1
            if gap >= MIN_GAP_FRAMES and current_event:
                raw_events.append(current_event)
                current_event = []
    if current_event:
        raw_events.append(current_event)

    valid_events = []
    for ev in raw_events:
        max_len = cur = 0
        for fname in ev:
            img = cv2.imread(os.path.join(INPUT_DIR, fname))
            if detect_event_frame(img):
                cur += 1
                max_len = max(max_len, cur)
            else:
                cur = 0
        if max_len >= MIN_EVENT_FRAMES:
            valid_events.append(ev)

    print(f"\n===== 事件统计 =====")
    print(f"📊 原始事件数量: {len(raw_events)}")
    print(f"✅ 有效事件数量: {len(valid_events)}")

    print("\n===== 事件级预测结果 =====")
    for idx, ev in enumerate(valid_events):
        if len(ev) <= 2:
            continue
        ev_core = ev[1:-1]
        pred, vote_detail = predict_event(model, ev_core, INPUT_DIR)
        if pred == -1:
            print(f"Event {idx+1}: ❌ 无有效预测")
            continue
        print(f"Event {idx+1}: 🎯 {CLASS_NAMES[pred]}, 投票={vote_detail}")

    

if __name__ == "__main__":
    np.random.seed(42)
    torch.manual_seed(42)
    main()

    
    w1 = np.load(FC1_WEIGHT_NPY).astype(np.int8)
    w2 = np.load(FC2_WEIGHT_NPY).astype(np.int8)
    w3 = np.load(FC3_WEIGHT_NPY).astype(np.int8)

    model = SNN_Paibox(w1, w2, w3)

    # model = Int8FCPart(weight1.T)
    mapper = pb.Mapper()
    mapper.build(model)
    graph_info = mapper.compile(core_estimate_only=False, weight_bit_optimization=True, grouping_optim_target="both")
    mapper.export(write_to_file=True, fp="./debug_260422/", format="bin", split_by_chip=False, use_hw_sim=True)

    print(graph_info.get('n_core_required'))  # 推荐用 get()，不存在时返回 None 而非报错

    





===== 加载模型 =====
✅ 模型加载完成

===== 读取事件帧（/Users/bytedance/Desktop/dvs_data/test_offline） =====
📁 有效帧数量: 486

===== 事件分割 =====

===== 事件统计 =====
📊 原始事件数量: 9
✅ 有效事件数量: 8

===== 事件级预测结果 =====
Event 1: 🎯 heitao, 投票={np.int64(0): np.int64(1), np.int64(1): np.int64(12), np.int64(3): np.int64(2)}
Event 2: 🎯 hongtao, 投票={np.int64(0): np.int64(1), np.int64(1): np.int64(2), np.int64(2): np.int64(16)}
Event 3: 🎯 meihua, 投票={np.int64(2): np.int64(2), np.int64(3): np.int64(11)}
Event 4: 🎯 fangpian, 投票={np.int64(0): np.int64(12), np.int64(1): np.int64(2), np.int64(3): np.int64(1), np.int64(4): np.int64(1)}
Event 5: 🎯 heitao, 投票={np.int64(1): np.int64(11), np.int64(2): np.int64(2)}
Event 6: 🎯 hongtao, 投票={np.int64(0): np.int64(2), np.int64(1): np.int64(4), np.int64(2): np.int64(9), np.int64(3): np.int64(1)}
Event 7: 🎯 meihua, 投票={np.int64(2): np.int64(1), np.int64(3): np.int64(15)}
Event 8: 🎯 fangpian, 投票={np.int64(0): np.int64(11), np.int64(1): np.int64(1), np.int64(2): np.int64(1)}
RoutingGroup: Rout

In [13]:
# 3，224，224
# 3，112，112
# 8，56，56
# 16，28，28
# 32，14，14

In [ ]:

import os
os.chdir("/Users/bytedance/Desktop/bishe")

import paibox as pb
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import transforms
from PIL import Image

from spikingjelly.activation_based import neuron, functional
from qat_layer import Quan_Linear

class Int8FCPart(pb.Network):
    def __init__(self, fc1_weight):
        super().__init__()
        weight0 = np.random.randint(low=-128, high=127, size=(32*14*14, 1024), dtype=np.int8)
        weight1 = np.random.randint(low=-128, high=127, size=(1024, 512), dtype=np.int8)
        weight2 = np.random.randint(low=-128, high=127, size=(512, 128), dtype=np.int8)
        weight3 = np.random.randint(low=-128, high=127, size=(128, 5), dtype=np.int8)

        self.in_neuron = pb.InputProj(input=None, shape_out=(32*14*14,))
        self.n0 = pb.LIF((1024,), threshold=1, reset_v=0)
        self.s1 = pb.FullConn(
            self.in_neuron,
            self.n0,
            weights=weight0,
            conn_type=pb.SynConnType.All2All
        )

        self.n1 = pb.LIF((512,), threshold=1, reset_v=0)
        self.s1 = pb.FullConn(
            self.n0,
            self.n1,
            weights=weight1,
            conn_type=pb.SynConnType.All2All
        )

       

        self.n2 = pb.LIF(128, threshold=int(1.0), reset_v=0)
        self.s2 = pb.FullConn(
            self.n1,
            self.n2,
            weights=weight2,
            conn_type=pb.SynConnType.All2All
        )

        self.n3 = pb.LIF(5, threshold=int(1.0), reset_v=0)
        self.s3 = pb.FullConn(
            self.n2,
            self.n3,
            weights=weight3,
            conn_type=pb.SynConnType.All2All
        )

        # self.n4 = pb.LIF(32, threshold=int(1.0), reset_v=0, tick_wait_start=2)
        # self.n5 = pb.LIF(2, threshold=int(1.0), reset_v=0, tick_wait_start=3)

        

        self.probe_out = pb.Probe(self.s2, attr="output")

        self.sim = pb.Simulator(self)




import os
# 把工作目录切换到项目根目录 bishe（根据你的实际路径修改）
os.chdir("/Users/bytedance/Desktop/bishe")

import paibox as pb
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models, transforms
from PIL import Image

from spikingjelly.activation_based import neuron, layer, functional

from qat_layer import Quan_Linear
from paibox.components.neuron.neurons import StoreVoltageNeuron as StoreVoltageNeuron
import numpy as np
FC2_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment/save_int8_new/fc2_weight_int8.npy"  # INT8 FC2权重
FC3_WEIGHT_NPY = "/Users/bytedance/Desktop/bishe/2_huase_augment/save_int8_new/fc3_weight_int8.npy"  # INT8 FC3权重

weight1 = np.load(FC2_WEIGHT_NPY).astype(np.int8)
weight2 = np.load(FC3_WEIGHT_NPY).astype(np.int8)

model = Int8FCPart(weight1.T)
mapper = pb.Mapper()
mapper.build(model)
graph_info = mapper.compile(core_estimate_only=False, weight_bit_optimization=True, grouping_optim_target="both")
mapper.export(write_to_file=True, fp="./debug_260404/", format="bin", split_by_chip=False, use_hw_sim=True)

print(graph_info.get('n_core_required'))  # 推荐用 get()，不存在时返回 None 而非报错





RoutingGroup: RoutingGroup_40 with 0 cores:
multicast axons: ['InputProj_11']
	CoreBlock_18 with 0 cores:
		LCN: 3
		FullConn_26: InputProj_11 -> LIF_31

RoutingGroup: RoutingGroup_41 with 0 cores:
multicast axons: ['LIF_31']
	CoreBlock_19 with 0 cores:
		LCN: 0
		FullConn_27: LIF_31 -> LIF_32

RoutingGroup: RoutingGroup_42 with 0 cores:
multicast axons: ['LIF_32']
	CoreBlock_20 with 0 cores:
		LCN: 0
		FullConn_28: LIF_32 -> LIF_33

RoutingGroup: RoutingGroup_43 with 0 cores:
multicast axons: ['LIF_33']
	CoreBlock_21 with 0 cores:
		LCN: 0
		FullConn_29: LIF_33 -> LIF_34

139
